# Guide 6 — Solving the Pre-Game Riddle (asking an AI)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

Before the game starts, both teams get a riddle, like *"I speak without a mouth..."*
There's no answer list to look it up in, so we ask a helper AI (a language model) to
solve it. Whoever says the right answer out loud first wins that bonus — the AI just
helps you answer fast.

All real code. The clever part is making sure a slow AI can never freeze the
program.


### How this guide fits in

**Depends on:** Guide 1 (loads the `bootcamp_ai` helper). **Used by:** Guide 12.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Setting up the question

We pick which AI helper to use and write the question in a way that asks for a short,
tidy answer.


In [ ]:
PREGAME_RIDDLE_MODEL = 'Smart Helper'  # bootcamp_ai friendly model name
PREGAME_RIDDLE_TIMEOUT_SECONDS = 20


def build_pregame_riddle_prompt(riddle_text):
    return (
        'You are a fast riddle-solving assistant for a live competition. '
        'A player was just given this riddle and needs the answer immediately '
        'so they can say it out loud before their opponent:\n\n'
        f'"{riddle_text}"\n\n'
        'Reply with ONLY compact JSON, no extra words, in exactly this shape:\n'
        '{"answer": "<short answer, one or two words>"}'
    )

### Reading the AI's answer

AIs don't always answer neatly. These helpers dig the actual answer out of the
reply, even if it's messy or has extra words around it.


In [ ]:
def _extract_json_object(reply_text):
    """Extract the first valid JSON object, including from fenced/prose replies."""
    text = str(reply_text or '')
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    raise ValueError(f'LLM reply did not contain valid JSON: {reply_text!r}')


def _fallback_answer_from_llm(reply_text):
    """Recover a short answer from common non-JSON LLM responses."""
    text = str(reply_text or '').strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.IGNORECASE).strip()
    match = re.search(r'["\']?answer["\']?\s*[:=]\s*["\']?([^"\'\n}\]]+)', text, re.IGNORECASE)
    if not match:
        match = re.search(r'\banswer\s+is\s+([^\n.!?]+)', text, re.IGNORECASE)
    if match:
        answer = match.group(1)
    else:
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        answer = lines[0] if lines else ''
    answer = re.sub(r'^(?:answer\s*[:=-]\s*)', '', answer, flags=re.IGNORECASE)
    answer = answer.strip().strip('`"\'{}[]').rstrip('.,;:').strip()
    if not answer:
        raise ValueError(f'LLM reply did not contain a usable answer: {reply_text!r}')
    return answer[:80].strip()

### Asking without getting stuck

This is the important trick: it asks the AI on a separate "thread" with a time limit,
so if the AI is slow, the game keeps running instead of freezing.


In [ ]:
def _prompt_llm_with_timeout(prompt_text, timeout_seconds):
    """Run the helper in a daemon thread so its 300s HTTP timeout cannot hang the notebook."""
    result_queue = queue.Queue(maxsize=1)

    def worker():
        try:
            reply = bootcamp_ai.prompt(
                prompt_text,
                enforce_cooldown=False,
                session='pregame_riddle',
            )
            result_queue.put(('ok', reply))
        except Exception as exc:
            result_queue.put(('error', exc))

    threading.Thread(target=worker, daemon=True, name='pregame-riddle-llm').start()
    try:
        status, payload = result_queue.get(timeout=float(timeout_seconds))
    except queue.Empty as exc:
        raise TimeoutError(f'LLM did not respond within {timeout_seconds} seconds.') from exc
    if status == 'error':
        raise payload
    return payload


def solve_pregame_riddle_with_llm(riddle_text, model=None):
    """Return a best-guess answer without allowing a stalled or malformed LLM
    response to hang the match client."""
    riddle_text = str(riddle_text).strip()
    if not riddle_text:
        raise ValueError('Riddle text is empty.')
    bootcamp_ai.use_model(model or PREGAME_RIDDLE_MODEL)
    reply_text = _prompt_llm_with_timeout(
        build_pregame_riddle_prompt(riddle_text),
        PREGAME_RIDDLE_TIMEOUT_SECONDS,
    )
    used_fallback = False
    try:
        parsed = _extract_json_object(reply_text)
        answer = str(parsed.get('answer') or '').strip()
        if not answer:
            raise ValueError('JSON answer was empty.')
    except (TypeError, ValueError, json.JSONDecodeError):
        answer = _fallback_answer_from_llm(reply_text)
        used_fallback = True
    return {
        'riddle': riddle_text,
        'answer': answer,
        'raw_reply': reply_text,
        'used_fallback': used_fallback,
    }

### Check yourself

1. Why do we put a time limit on the AI's answer?
2. Why do we need a backup way to read the answer if the neat version fails?
